# Data Transformation

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, RobustScaler,LabelEncoder
import joblib

In [2]:
train_df = pd.read_csv("../../data/split/train.csv")
val_df = pd.read_csv("../../data/split/validation.csv")
test_df = pd.read_csv("../../data/split/test.csv")
x_train = train_df.drop("diabetes", axis=1)
y_train = train_df["diabetes"]

x_val = val_df.drop("diabetes", axis=1)
y_val = val_df["diabetes"]

x_test = test_df.drop("diabetes", axis=1)
y_test = test_df["diabetes"]

## Feature Discretization

---

### HbA1c Level → `hba1c_category`

Measures average blood sugar over the past 2–3 months.

| Category    | Range         | Clinical Meaning                        |
|-------------|---------------|-----------------------------------------|
| Normal      | < 5.7%        | Healthy glucose metabolism              |
| Prediabetes | 5.7% – 6.4%   | Elevated risk, not yet diabetic         |
| Diabetes    | ≥ 6.5%        | Meets diagnostic threshold for diabetes |

---

### BMI → `bmi_category`

Body Mass Index — ratio of weight to height squared (kg/m²).

| Category       | Range         | Clinical Meaning                              |
|----------------|---------------|-----------------------------------------------|
| Underweight    | < 18.5        | May indicate malnutrition or metabolic issues |
| Healthy Weight | 18.5 – 24.9   | Optimal weight range                          |
| Overweight     | 25.0 – 29.9   | Increased risk of metabolic disorders         |
| Obesity        | ≥ 30.0        | High risk of diabetes, hypertension, CVD      |

---

### Blood Glucose Level → `blood_glucose_category`

Fasting blood glucose measured in mg/dL.

| Category          | Range          | Clinical Meaning                            |
|-------------------|----------------|---------------------------------------------|
| Normal            | 70 – 139 mg/dL | Healthy fasting glucose                     |
| Prediabetes       | 140 – 199 mg/dL| Impaired glucose tolerance                  |
| Possible Diabetes | ≥ 200 mg/dL    | Meets diagnostic threshold, requires workup |

---

### Notes

- `right=False` ensures boundary values fall into the **higher-risk category**
  - e.g. `HbA1c = 6.5` → **Diabetes**, not Prediabetes
  - e.g. `BMI = 30.0` → **Obesity**, not Overweight
- All labels follow **ordinal risk ordering** — suitable for mapping to integers if needed

In [3]:
def discretize(df):
    df["hba1c_category"] = pd.cut(
            df['HbA1c_level'],
            bins=[0, 5.7, 6.5, float("inf")],
            labels=["Normal", "Prediabetes", "Diabetes"],
            right= False
        )
    
    df['bmi_category'] = pd.cut(
            df['bmi'],
            bins=[0, 18.5, 25, 30, float("inf")],
            labels=["Underweight", "Healthy Weight", "Overweight", "Obesity"],
            right= False
        )
    
    df['blood_glucose_category'] = pd.cut(
            df['blood_glucose_level'],
            bins=[70, 140, 200, float("inf")],
            labels=["Normal", "Prediabetes", "Possible Diabetes"],
            right= False
        )
    
    return df

## Feature Encoding 
### Gender Binary Encoding 
- Converts categorical gender to numeric binary
- Female = 0 | Male = 1

In [4]:
x_train["gender"] = x_train["gender"].replace({"Female": 0, "Male": 1})
x_val["gender"] = x_val["gender"].replace({"Female": 0, "Male": 1})
x_test["gender"] = x_test["gender"].replace({"Female": 0, "Male": 1})

C:\Users\Dell\AppData\Local\Temp\ipykernel_34140\2215808311.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_train["gender"] = x_train["gender"].replace({"Female": 0, "Male": 1})
C:\Users\Dell\AppData\Local\Temp\ipykernel_34140\2215808311.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_val["gender"] = x_val["gender"].replace({"Female": 0, "Male": 1})
C:\Users\Dell\AppData\Local\Temp\ipykernel_34140\2215808311.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To

### Ordinal Encoding for Discretized Categories
- Maps clinically ordered string categories to integers
- Preserves natural risk ordering: lowest risk = 0 → highest risk = n

In [5]:
def ordinal_encode_categories(df):

    hba1c_mapping = {
        "Normal"      : 0,
        "Prediabetes" : 1,
        "Diabetes"    : 2
    }

    bmi_mapping = {
        "Underweight"    : 0,
        "Healthy Weight" : 1,
        "Overweight"     : 2,
        "Obesity"        : 3
    }

    blood_glucose_mapping = {
        "Normal"            : 0,
        "Prediabetes"       : 1,
        "Possible Diabetes" : 2
    }

    df["hba1c_category"]         = df["hba1c_category"].map(hba1c_mapping).astype(int)
    df["bmi_category"]           = df["bmi_category"].map(bmi_mapping).astype(int)
    df["blood_glucose_category"] = df["blood_glucose_category"].map(blood_glucose_mapping).astype(int)

    return df

## Feature Engineering

### Arithmetic Interaction Features

Captures the **combined effect** of two clinical variables that a model may miss
when evaluating each feature in isolation. Computed as a simple multiplication
of the two source features.

| Feature | Formula | Clinical Rationale |
|---|---|---|
| `glucose_hba1c_interaction` | `blood_glucose_level × HbA1c_level` | Combines short-term and long-term glucose markers — their joint elevation is a stronger diabetes signal than either alone |
| `age_hba1c_interaction` | `age × HbA1c_level` | Captures how HbA1c-related risk compounds with age, reflecting age-dependent metabolic decline |
| `age_bmi_interaction` | `age × bmi` | Reflects how excess body weight becomes increasingly harmful to glucose metabolism at older ages |
| `bmi_hba1c_interaction` | `bmi × HbA1c_level` | Encodes the joint burden of obesity and poor long-term glucose control, a key metabolic syndrome signal |
| `age_glucose_interaction` | `age × blood_glucose_level` | Captures how elevated blood glucose carries greater diagnostic weight in older age groups |

---

### Binary Flags & Composite Risk Features

Converts clinical thresholds and domain knowledge into **binary signals (0/1)**,
making categorical risk boundaries explicitly available to the model.

| Feature | Condition | Clinical Rationale |
|---|---|---|
| `high_hba1c_flag` | `HbA1c >= 6.5` | Flags patients at or above the WHO diagnostic threshold for diabetes |
| `senior_flag` | `age >= 60` | Flags patients in the age group with the highest prevalence of type 2 diabetes |
| `cardio_risk_flag` | `hypertension == 1` OR `heart_disease == 1` | Flags patients with any cardiovascular comorbidity, which is strongly associated with insulin resistance and diabetes risk |

In [6]:
def add_engineered_features(df):
    # arithimetic features     
    df["glucose_hba1c_interaction"] = df["blood_glucose_level"] * df["HbA1c_level"]
    df["age_hba1c_interaction"] = df["age"] * df["HbA1c_level"]
    df["age_bmi_interaction"] = df["age"] * df["bmi"]
    df["bmi_hba1c_interaction"] = df["bmi"] * df["HbA1c_level"]
    df["age_glucose_interaction"] = df["age"] * df["blood_glucose_level"]
    
    # Boolean and Logical Combinations
    df["high_hba1c_flag"] = (df["HbA1c_level"] >= 6.6).astype(int)
    df["senior_flag"] = (df["age"] >= 60).astype(int)
    df["cardio_risk_flag"] = ((df["hypertension"] == 1) | (df["heart_disease"] == 1)).astype(int)

    return df

## Data Normalization

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("smoking_ohe", OneHotEncoder(handle_unknown="ignore",drop="first"), ["smoking_history"]),
        ("age_minmax", MinMaxScaler(), ["age"]),
        ("robust_features", RobustScaler(), ["bmi",
                                              "HbA1c_level",
                                              "blood_glucose_level",
                                              "glucose_hba1c_interaction",
                                              "age_hba1c_interaction",
                                              "age_bmi_interaction",
                                              "bmi_hba1c_interaction",
                                              "age_glucose_interaction"])
    ],
    remainder="passthrough", # to keep the rest of the columns unchanged
    verbose_feature_names_out=False  # to prevent adding the transformer name as a prefix to the feature names
    # smoking_history_No Info insted of smoking_ohe__smoking_history_No Info
)
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

## Transform Training data 

In [8]:
x_train_featured = add_engineered_features(x_train)
x_train_processed = preprocessor.fit_transform(x_train_featured) # returns numpy array not Pandas DataFrame so we need to convert it back to DataFrame
joblib.dump(preprocessor, "preprocessor.pkl")
feature_names = preprocessor.get_feature_names_out()
x_train_processed = pd.DataFrame(
    x_train_processed,
    columns=feature_names,
    index=x_train.index
)
x_train_processed.head()

smoking_history_current smoking_history_ever smoking_history_former  \
0                     0.0                  0.0                    0.0   
1                     0.0                  0.0                    1.0   
2                     0.0                  0.0                    1.0   
3                     0.0                  1.0                    0.0   
4                     0.0                  0.0                    0.0   

  smoking_history_never smoking_history_not current       age       bmi  \
0                   0.0                         0.0  0.612112 -0.637209   
1                   0.0                         0.0  0.737237  0.818605   
2                   0.0                         0.0  0.399399 -0.229457   
3                   0.0                         0.0  0.874875  1.485271   
4                   1.0                         0.0  0.687187 -0.395349   

  HbA1c_level blood_glucose_level glucose_hba1c_interaction  \
0         0.0           -0.677966                 -0.459103   
1         0.5                 0.0                  0.411609   
2         0.0            0.254237                  0.382586   
3         0.5            0.254237                  0.668865   
4   -1.642857            1.016949                  -0.14248   

  age_hba1c_interaction age_bmi_interaction bmi_hba1c_interaction  \
0              0.289448           -0.056612             -0.232854   
1              0.770087            0.658534              1.018393   
2             -0.187803           -0.339001              0.014118   
3              1.116167             1.25859              1.470922   
4             -0.154405            0.148131             -1.008759   

  age_glucose_interaction gender hypertension heart_disease high_hba1c_flag  \
0               -0.081827      0            0             0               0   
1                0.557564      0            0             0               0   
2               -0.070409      1            0             0               0   
3                1.050428      0            0             0               0   
4                1.078972      1            0             0               0   

  senior_flag cardio_risk_flag  
0           0                0  
1           0                0  
2           0                0  
3           1                0  
4           0                0

## Transform Validation data 

In [9]:
x_val = add_engineered_features(x_val)
preprocessor = joblib.load("preprocessor.pkl")
x_val_processed = preprocessor.transform(x_val)
x_val_processed = pd.DataFrame(
    x_val_processed,
    columns=feature_names,   # same feature names from train
    index=x_val.index
)

x_val_processed.head()

,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,age_hba1c_interaction,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag
0,0.0,0.0,1.0,0.0,0.0,0.974975,1.035659,-0.071429,-0.847458,-0.635884,1.065828,1.321361,0.725342,0.321598,0,0,0,0,1,0
1,0.0,0.0,0.0,1.0,0.0,0.624625,-1.127132,-0.571429,0.338983,0.121372,0.123911,-0.179235,-0.789295,0.508088,0,0,0,0,0,0
2,0.0,0.0,0.0,1.0,0.0,0.662162,0.0,0.571429,-0.847458,-0.422164,0.60697,0.226013,0.506962,-0.106565,0,0,0,1,0,0
3,0.0,0.0,0.0,0.0,0.0,0.787287,0.0,-0.071429,0.322034,0.401847,0.651985,0.47455,0.108866,0.891912,1,1,0,0,1,1
4,0.0,0.0,0.0,1.0,0.0,0.787287,2.344186,0.142857,0.338983,0.543536,0.743466,1.341121,1.710381,0.903901,0,0,0,0,1,0


## Transform Cleaned Dataset For EDA stage

In [10]:
cleaned = pd.read_csv("../../data/cleaned/cleaned_1.csv")
cleaned_and_binned = discretize(cleaned)
cleaned_binned_engineered = add_engineered_features(cleaned_and_binned)

cleaned_binned_engineered.to_csv("../../data/EDA/transformed_for_eda.csv", index=False)

In [ ]:
train_processed_df = x_train_processed.copy()
train_processed_df["diabetes"] = y_train.values

val_processed_df = x_val_processed.copy()
val_processed_df["diabetes"] = y_val.values

train_processed_df.to_csv("../../data/preprocessed/train_preprocessed.csv", index=False)
val_processed_df.to_csv("../../data/preprocessed/validation_preprocessed.csv", index=False)
